<a href="https://colab.research.google.com/github/AnuroopVJ/GPT-Style-Transformer/blob/main/llm_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -Uq tiktoken torch matplotlib

In [ ]:

import matplotlib.pyplot as plt
import pandas as pd
import os

In [ ]:
!pip install datasets

In [ ]:
!pip install tiktoken

In [ ]:
from datasets import load_dataset

dataset = load_dataset("wikimedia/wikipedia", "20231101.en", split="train")

In [ ]:
print(dataset[0]['text'])

In [ ]:
import tiktoken
encoder = tiktoken.get_encoding("gpt2")
EOT_TOKEN = encoder.eot_token  # <|endoftext|> token ID (50256)
encoder

In [ ]:
text_encode = dataset[:100000]["text"]
text_encode

In [ ]:
encoded_list_full = []
for entry in text_encode:
    encoded_list = encoder.encode(entry)
    encoded_list_full.extend(encoded_list)

In [ ]:
encoded_list_full

In [ ]:
vocab_size = encoder.n_vocab
vocab_size

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [ ]:
token_tensor = torch.tensor(encoded_list_full, dtype=torch.long)

In [ ]:
# split to blocks
block_size = 64
step = 32
windows = token_tensor.unfold(0, block_size + 1, step)
x = windows[:, :-1]
y = windows[:,1:]

In [ ]:
x

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
dataset_for_training = TensorDataset(x, y)
train_dataloader = DataLoader(dataset_for_training, batch_size=32, shuffle=True)


In [ ]:

train_features, train_labels = next(iter(train_dataloader))
print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels.size()}")


In [ ]:
class LanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embeddings = nn.Embedding(VOCAB_SIZE, DIM)
        self.position_embeddings = nn.Embedding(block_size, DIM)
        self.attn = nn.MultiheadAttention(DIM, num_heads=4, batch_first=True)
        self.norm1 = nn.LayerNorm(DIM)

        # FFN layer
        self.ffn = nn.Sequential(
            nn.Linear(DIM, 4 * DIM), # Expand dimension
            nn.GELU(),
            nn.Linear(4 * DIM, DIM)  # Project back to DIM
        )
        self.norm2 = nn.LayerNorm(DIM)
        self.lm_head = nn.Linear(DIM, VOCAB_SIZE)

    def forward(self, idx):
        B, T = idx.shape
        x = self.token_embeddings(idx) + self.position_embeddings(torch.arange(T, device=idx.device))

        # Attention block with pre-norm
        attn_input = self.norm1(x)
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=idx.device)
        attn_output, _ = self.attn(attn_input, attn_input, attn_input, attn_mask=mask, is_causal=True)
        x = x + attn_output  # Residual connection after attention

        # FFN block with pre-norm
        ffn_input = self.norm2(x)
        ffn_output = self.ffn(ffn_input)
        x = x + ffn_output # Residual connection after FFN

        return self.lm_head(x)

In [ ]:
VOCAB_SIZE = 50257
DIM = 64

In [ ]:
# test LanguageModel works with 1 batch
model = LanguageModel()
xb, yb = next(iter(train_dataloader))
l = model(xb)
print(l.shape)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)  # confirm GPU is being used

In [ ]:
# Full training cell
import time

model = LanguageModel().to(device)
model = torch.compile(model)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler('cuda')

total_batches = len(train_dataloader)
start = time.time()

for batch_idx, (xb, yb) in enumerate(train_dataloader):
    xb, yb = xb.to(device), yb.to(device)          # critical - data on GPU
    optimizer.zero_grad(set_to_none=True)

    with torch.amp.autocast('cuda'):
        logits = model(xb)
        B, T, C = logits.shape
        loss = F.cross_entropy(logits.view(B*T, C), yb.view(B*T))

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    if batch_idx % 100 == 0 and batch_idx > 0:
        rate = (time.time() - start) / batch_idx
        eta = rate * (total_batches - batch_idx)
        print(f"[{batch_idx}/{total_batches}] loss {loss.item():.4f} | ETA {eta/60:.1f}m")

In [ ]:
# Training Loooop
import torch.nn.functional as F
model = LanguageModel()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
for epoch in range(1):
    for batch_idx, (xb, yb) in enumerate(train_dataloader):
        logits = model(xb)
        B, T, C = logits.shape
        loss = F.cross_entropy(logits.view(B*T, C), yb.view(B*T))
        print(loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [ ]:
print(f"Number of batches in train_dataloader: {len(train_dataloader)}")

In [ ]:
torch.save(model.state_dict(), 'llm.pth')

In [ ]:
model = LanguageModel().to(device)
model = torch.compile(model) # Compile the model before loading the state_dict
model.load_state_dict(torch.load('/content/llm.pth', map_location=device))
model.eval()

In [ ]:
def generate(model, prompt, max_new_tokens=100, temperature=1.0):
    model.eval()
    tokens = encoder.encode(prompt)
    tokens = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)  # (1, T)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            # crop to block_size if too long
            tokens_cropped = tokens[:, -block_size:]

            logits = model(tokens_cropped)         # (1, T, vocab)
            logits = logits[:, -1, :]              # last token only (1, vocab)
            logits = logits / temperature          # temperature scaling

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)  # sample
            tokens = torch.cat([tokens, next_token], dim=1)

    return encoder.decode(tokens[0].tolist())

# run it
print(generate(model, prompt="France is", max_new_tokens=200, temperature=0.8))

In [ ]:
print(generate(model, prompt="The history of", max_new_tokens=200, temperature=0.8))

In [ ]:
# ── Test Cell ──────────────────────────────────────────────────────────────────
prompts = [
    "The history of",
    "In the year",
    "The United States",
    "Science and technology",
]

model.eval()
print("=" * 60)
for prompt in prompts:
    output = generate(model, prompt, max_new_tokens=80, temperature=0.8)
    print(f"PROMPT: {prompt!r}")
    print(f"OUTPUT: {output}")
    print("-" * 60)